# Tutorial 30 — Load a model and chat

[Tutorial 20](./20_huggingface_models.ipynb) showed how QAChat finds model
files in the shared Hugging Face cache. This tutorial loads one chat model,
sends two messages, demonstrates conversation history, and then releases the
model.

The model weights, computational backend, and generation settings are kept as
separate choices: the backend decides how the model runs, the model ID selects
the weights, and the generation settings control how output is produced.

Retrieval-augmented generation (RAG) is not used yet. The following tutorials
first build the repository-processing and retrieval layers that will later be
combined with this chat model.


## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import platform

print(f"Computer system: {platform.system()}")
print(f"Computer machine: {platform.machine()}")


In [ ]:
import qachat


## 2. Inspect cached model candidates

This reads Hugging Face's shared cache. Each printed entry is a model ID, but the
list is only an inventory: it can contain chat, embedding, image, or otherwise
incompatible models. The next section therefore chooses QAChat's canonical chat
model for the resolved backend rather than assuming that the first cached model
is suitable.


In [ ]:
models = qachat.available_models()
if models:
    print("Model IDs found in the Hugging Face cache:")
    for number, model in enumerate(models, start=1):
        size_gb = model["size_bytes"] / 1_000_000_000
        print(f"{number}. {model['model_id']} ({size_gb:.2f} GB)")
else:
    print("No models are cached yet.")


## 3. Choose the backend and model

`BACKEND = "auto"` asks QAChat to resolve the appropriate local runtime. The
selected backend then determines which canonical chat-model default is used.


In [ ]:
REQUESTED_BACKEND = qachat.BACKEND_AUTO

SELECTED_BACKEND = qachat.choose_backend(REQUESTED_BACKEND)

print(f"Requested backend: {REQUESTED_BACKEND}")
print(f"Resolved backend: {SELECTED_BACKEND}")


You can replace `MODEL_ID` with another compatible model when experimenting.
To use a cached model, copy its exact model ID from the list above, for example, `MODEL_ID = "Qwen/Qwen3-4B-MLX-4bit"`.

The other variables control generation rather than model selection:

- `THINKING` is passed to compatible chat templates.

- `TEMPERATURE = 0.0` requests repeatable greedy generation.

- `MAX_NEW_TOKENS = None` adds no QAChat length cap; generation uses the model's remaining context. Set a positive number to use less output tokens.

- `ALLOW_DOWNLOAD = True` and files are missing, Hugging Face downloads them to its normal cache during the loading cell.


In [ ]:
MODEL_ID = (
    qachat.DEFAULT_CHAT_MODEL_MLX
    if SELECTED_BACKEND == "mlx"
    else qachat.DEFAULT_CHAT_MODEL_TORCH
)
THINKING = False
TEMPERATURE = 0.0
MAX_NEW_TOKENS = None
ALLOW_DOWNLOAD = True

print(f"Chat model: {MODEL_ID}")


## 4. Load the selected model

This is the first cell in the tutorial series that loads language-model weights
into memory. `ChatSession` owns the model lifecycle and stores the conversation
messages used by later replies. Loading can require several gigabytes of memory.


In [ ]:
if "session" in globals():
    session.close()  # It closes an earlier session before creating a new one.


In [ ]:
session = qachat.ChatSession(
    MODEL_ID,
    backend=SELECTED_BACKEND,
    thinking=THINKING,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    system_prompt="Answer clearly and briefly.",
    allow_download=ALLOW_DOWNLOAD,
)


In [ ]:
print(f"Loaded model: {session.model_id}")
print(f"Backend: {session.backend}")
print(f"Device: {session.device}")


## 5. Chat with conversation history

The first call sends one user message. The second call uses the same session, so
its prompt also contains the earlier user/assistant exchange. This demonstrates
conversation state without any repository retrieval.


In [ ]:
question_1 = "Hello, I am Iok-ui! Please, greet me in one short sentence."

print(f"QUESTION 1: {question_1}")


In [ ]:
answer_1 = session.reply(question_1)

print(f"ANSWER 1: {answer_1}")


Because the same `ChatSession` remains open, the second message receives the
first user message and assistant reply as conversation history. The test asks
the model to recover a name from that earlier turn.


In [ ]:
question_2 = "Copy the name from my first message. Answer with the name only."

print(f"QUESTION 2: {question_2}")


In [ ]:
answer_2 = session.reply(question_2)

print(f"ANSWER 2: {answer_2}")


## 6. Release the model

Closing releases this session's model memory and clears its messages. Cached model files remain available for later use.


In [ ]:
session.close()

print("Chat closed.")


## Continue the tutorial series

**Previous:** [Tutorial 20 — Work with Hugging Face models](./20_huggingface_models.ipynb)

**Next:** [Tutorial 40 — Select repository files](./40_repository_files.ipynb)
